# 🔬 MorphoBricks preprocessing\n\nPrepare a microscopy experiment for the other MorphoBricks bricks. Run the two cells below, enter the experiment path in the app, and follow only the confirmation prompts that appear. The processing itself is handled by the packaged MorphoBricks preprocessing pipeline.\n\n> **Google Colab:** keep the experiment and output on Google Drive under `/content/drive/MyDrive/`.\n>\n> **JupyterLab / LabConstrictor:** use normal local paths. The same widget app and pipeline are used in both environments.\n

In [ ]:
# @title Start MorphoBricks preprocessing
import platform
import subprocess
import sys
from importlib import import_module, invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

# LabConstrictor reads these two variables to track notebook versions.
current_version = "0.4.0rc6"
notebook_name = "MorphoBricks_Preprocessing"

# Remote notebooks install one immutable package revision. RELEASE_TAG is the human-facing
# release name; RELEASE_REF is the exact source used by Colab and downloaded notebooks.
RELEASE_TAG = "v0.4.0rc6"
RELEASE_REF = "d6930b80f26bce1b12762940d3c9ed44b038949f"
REPOSITORY = "https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
EXTRAS = "notebook"

if "google.colab" in sys.modules:
    from google.colab import drive

    print("Google Colab: connecting Google Drive...", flush=True)
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        raise RuntimeError(
            "Google Drive could not be connected. Allow Drive access, then run this cell again."
        ) from exc
    EXTRAS = "notebook,zarr"
else:
    print(
        f"Local Jupyter: Python {platform.python_version()} on {platform.platform()}\n"
        f"Working folder: {Path.cwd()}",
        flush=True,
    )


def _missing_requirements() -> list[str]:
    """Return notebook requirements this kernel cannot import."""
    required = (
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
        *(("zarr",) if "zarr" in EXTRAS.split(",") else ()),
    )
    return [name for name in required if find_spec(name) is None]


def _start_preprocessing_notebook() -> object:
    """Prepare this kernel and return the notebook interface."""
    if sys.version_info < (3, 11):
        raise RuntimeError("Select a Python 3.11 or newer kernel, then run this cell again.")

    current = Path.cwd().resolve()
    checkout = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "pyproject.toml").is_file()
            and (folder / "src/morphobricks_preprocessing/__init__.py").is_file()
        ),
        None,
    )

    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("morphobricks_preprocessing")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel already loaded another copy of MorphoBricks preprocessing. "
                "Restart the kernel, then run this cell again."
            )
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        install_target = ["-e", f"{checkout}[{EXTRAS}]"]
        origin = f"local checkout at {checkout}"
    else:
        install_target = [
            f"morphobricks-preprocessing[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_REF}"
        ]
        try:
            installed = version("morphobricks-preprocessing")
        except PackageNotFoundError:
            installed = None
        origin = (
            f"installed package {installed}"
            if installed == current_version
            else f"pinned {RELEASE_TAG} candidate ({RELEASE_REF[:12]})"
        )

    missing = _missing_requirements()
    try:
        installed = version("morphobricks-preprocessing")
    except PackageNotFoundError:
        installed = None
    if checkout is None and installed != current_version:
        if "morphobricks_preprocessing" not in missing:
            missing.append("morphobricks-preprocessing " + current_version)

    if missing:
        print(f"Installing notebook requirements ({', '.join(missing)})...", flush=True)
        try:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--disable-pip-version-check",
                    *install_target,
                ],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output and internet access, then run this cell again."
            ) from exc
        invalidate_caches()

    remaining = _missing_requirements()
    if remaining:
        raise RuntimeError(
            "Installation finished, but this kernel still cannot import "
            + ", ".join(remaining)
            + ". Restart the kernel and run this cell again."
        )

    try:
        interface = import_module("morphobricks_preprocessing.notebook_ui")
    except ImportError as exc:
        raise RuntimeError(
            "MorphoBricks preprocessing was found, but its notebook interface could not load."
        ) from exc

    package_version = version("morphobricks-preprocessing")
    widget_version = version("ipywidgets")
    normalization_version = version("morphobricks-normalization")
    print(
        f"Ready. MorphoBricks preprocessing {package_version}; "
        f"normalization {normalization_version}; ipywidgets {widget_version}.\n"
        f"Notebook {notebook_name} {current_version}; source: {origin}.",
        flush=True,
    )
    return interface.PreprocessingNotebook()


notebook = _start_preprocessing_notebook()


## Preprocessing app\nThe app checks the dataset, asks only for decisions that cannot be inferred safely, then computes and records the preprocessing outputs. Detailed reference documentation stays in the user guide rather than in the notebook.\n

In [ ]:
# @title Open the preprocessing app
from IPython.display import display

display(notebook.app())
